# Regressão Linear Múltipla — Exemplo 04

Prever o **valor de venda real** usando:
- valor de custo
- valor de venda cadastrado
- quantidade vendida
- categoria do produto

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

## 1. Conexão com o Banco de Dados PostgreSQL

In [ ]:
usuario = "datadt_data_analytics"
senha = "DataAnalytics$100"
host = "postgresql-datadt.alwaysdata.net"
porta = "5432"
banco = "datadt_digital_corporativo"

engine = create_engine(
    f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{banco}"
)

## 2. Consulta SQL

Cada linha representa um item vendido.

| Variável | Coluna |
|----------|--------|
| Y | valor_venda_real |
| X1 | valor_custo |
| X2 | valor_venda_cadastrado |
| X3 | quantidade |
| X4 | categoria |

In [ ]:
sql = """
SELECT 
    inf.id AS id_item,
    p.id AS id_produto,
    p.nome AS produto,
    c.descricao AS categoria,
    p.valor_custo,
    p.valor_venda AS valor_venda_cadastrado,
    inf.quantidade,
    inf.valor_venda_real
FROM vendas.item_nota_fiscal inf
JOIN vendas.produto p 
    ON p.id = inf.id_produto
JOIN vendas.categoria c 
    ON c.id = p.id_categoria
WHERE p.valor_custo IS NOT NULL
  AND p.valor_venda IS NOT NULL
  AND inf.valor_venda_real IS NOT NULL
  AND inf.quantidade IS NOT NULL
  AND p.valor_custo > 0
  AND p.valor_venda > 0
  AND inf.valor_venda_real > 0
  AND inf.quantidade > 0
ORDER BY inf.id;
"""

## 3. Carregando os Dados

In [ ]:
df = pd.read_sql(sql, engine)

print("Primeiras linhas da base:")
print(df.head())

print("\nInformações da base:")
print(df.info())

print("\nResumo estatístico:")
print(
    df[[
        "valor_custo",
        "valor_venda_cadastrado",
        "quantidade",
        "valor_venda_real",
    ]].describe()
)

print("\nCategorias encontradas:")
print(df["categoria"].value_counts())

## 4. Tratamento Básico dos Dados

In [ ]:
df = df.dropna(
    subset=[
        "categoria",
        "valor_custo",
        "valor_venda_cadastrado",
        "quantidade",
        "valor_venda_real",
    ]
)

df = df[
    (df["valor_custo"] > 0)
    & (df["valor_venda_cadastrado"] > 0)
    & (df["quantidade"] > 0)
    & (df["valor_venda_real"] > 0)
]

print("Quantidade de registros após tratamento:")
print(len(df))

## 5. Definindo X e Y

In [ ]:
X = df[[
    "valor_custo",
    "valor_venda_cadastrado",
    "quantidade",
    "categoria",
]]

y = df["valor_venda_real"]

## 6. Variáveis Numéricas e Categóricas

In [ ]:
variaveis_numericas = [
    "valor_custo",
    "valor_venda_cadastrado",
    "quantidade",
]

variaveis_categoricas = [
    "categoria",
]

## 7. Pré-processamento

A `categoria` é texto, por isso usamos `OneHotEncoder` para transformá-la em colunas numéricas binárias.

In [ ]:
pre_processador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            variaveis_categoricas,
        ),
        (
            "numericas",
            "passthrough",
            variaveis_numericas,
        ),
    ]
)

## 8. Criando o Pipeline

In [ ]:
modelo = Pipeline(
    steps=[
        ("pre_processador", pre_processador),
        ("regressao", LinearRegression()),
    ]
)

## 9. Divisão em Treino e Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

## 10. Treinando o Modelo

In [ ]:
modelo.fit(X_train, y_train)

## 11. Fazendo Previsões

In [ ]:
y_pred = modelo.predict(X_test)

resultado = pd.DataFrame(
    {
        "valor_custo": X_test["valor_custo"],
        "valor_venda_cadastrado": X_test["valor_venda_cadastrado"],
        "quantidade": X_test["quantidade"],
        "categoria": X_test["categoria"],
        "valor_real": y_test,
        "valor_previsto": y_pred,
    }
)

print("Comparação entre valor real e valor previsto:")
print(resultado.head(10))

## 12. Avaliação do Modelo

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Métricas de avaliação:")
print(f"MAE  - Erro médio absoluto: {mae:.2f}")
print(f"MSE  - Erro quadrático médio: {mse:.2f}")
print(f"R²   - Coeficiente de determinação: {r2:.4f}")

## 13. Coeficientes do Modelo

In [ ]:
regressao = modelo.named_steps["regressao"]
pre_processador_treinado = modelo.named_steps["pre_processador"]

nomes_variaveis_categoricas = (
    pre_processador_treinado
    .named_transformers_["categoricas"]
    .get_feature_names_out(variaveis_categoricas)
)

nomes_variaveis = list(nomes_variaveis_categoricas) + variaveis_numericas

coeficientes = pd.DataFrame(
    {
        "variavel": nomes_variaveis,
        "coeficiente": regressao.coef_,
    }
)

print("Intercepto do modelo:")
print(f"{regressao.intercept_:.2f}")

print("\nCoeficientes do modelo:")
print(coeficientes.sort_values(by="coeficiente", ascending=False))

## 14. Gráfico: Valor Real x Valor Previsto

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(y_test, y_pred, alpha=0.5, label="Itens vendidos")

plt.xlabel("Valor real de venda")
plt.ylabel("Valor previsto de venda")
plt.title("Regressão Linear Múltipla - Valor Real x Valor Previsto")
plt.legend()
plt.grid(True)

plt.savefig(
    "grafico_regressao_exemplo04_real_vs_previsto.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print("Gráfico salvo em: grafico_regressao_exemplo04_real_vs_previsto.png")

## 15. Gráfico: Análise dos Erros

In [ ]:
erros = y_test - y_pred

plt.figure(figsize=(10, 6))

plt.scatter(y_pred, erros, alpha=0.5, label="Erros")
plt.axhline(y=0, linestyle="--", label="Erro zero")

plt.xlabel("Valor previsto")
plt.ylabel("Erro")
plt.title("Análise dos Erros da Regressão")
plt.legend()
plt.grid(True)

plt.savefig(
    "grafico_regressao_exemplo04_erros.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print("Gráfico salvo em: grafico_regressao_exemplo04_erros.png")

## 16. Simulação de Previsão

> **Importante:** troque a categoria abaixo por uma que exista na sua base.

In [ ]:
print("Categorias disponíveis na base:")
print(df["categoria"].unique())

novo_item = pd.DataFrame(
    {
        "valor_custo": [100],
        "valor_venda_cadastrado": [250],
        "quantidade": [2],
        "categoria": ["Móveis"],
    }
)

valor_estimado = modelo.predict(novo_item)

print("\nSimulação:")
print("Dados do novo item:")
print(novo_item)
print(f"\nValor de venda real estimado: R$ {valor_estimado[0]:.2f}")